# Diff-PGD Evaluation


### Cell Overview
| Cell | Purpose |
|------|----------|
| 1 | Imports |
| 2 | Config dict |
| 3 | Load classifier |
| 4 | Load SD 1.5 + ControlNet + LCM scheduler + OSCP LoRA |
| 4b | Load guided diffusion model (Diff-PGD surrogate) |
| 4c | DenoisedClassifier (SDEdit wrapper) |
| 5 | Helper functions (seed_everything, lcm_lora_denoise, generate_x_adv_diff_pgd) |
| 6 | Test dataset + DataLoader |
| 7 | Evaluation loop |
| 8 | Print metrics |
| 9 | Save CSV |


## Cell 1 — Imports

In [1]:
import os
import time
import random
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
import cv2
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import DataLoader

# Diffusion pipeline components
from diffusers import (
    LCMScheduler,
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
)

# Guided diffusion model (Diff-PGD surrogate)
from load_dm import get_imagenet_dm_conf

# Adversarial attacks
from advertorch.attacks import LinfPGDAttack

# ---------- Local modules ----------
from archs import get_archs
from dataset import get_dataset
from attack_tools import gen_pgd_confs

# Image I/O helpers from utils.py
from utils import si, mp

print("Imports OK")


d:\MLSEC-Final\CSC592-MLSEC-InstantPure\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0427 12:34:28.519000 13740 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "d:\MLSEC-Final\CSC592-MLSEC-InstantPure\.venv\Lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
    ^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'triton'


Imports OK


## Cell 2 — Config

In [ ]:
config = dict(
    # model/checkpoint paths
    # SD 1.5 base
    pretrained_model  = "stable-diffusion-v1-5/stable-diffusion-v1-5",
    # Canny ControlNet from HuggingFace Hub (or local cache)
    controlnet_model  = "lllyasviel/sd-controlnet-canny",
    # Path to the OSCP-trained LoRA weights directory (output of exp_train_lora.ipynb)
    lora_input_dir    = "logs/OSCP/unet_lora",
    # Diffusion scheduler type: "LCM" only
    scheduler_type    = "LCM",
    output_dir        = "output_final/",

    # purification parameters
    num_inference_step = 1,
    # strength: fraction of forward diffusion noise added before denoising
    # 0 = no change, 1 = fully re-sampled image
    strength           = 0.4,
    guidance_scale     = 1.0,
    control_scale      = 0.8,    # ControlNet conditioning weight

    # evaluation parameters
    classifier         = "resnet50",
    num_validation_set = 5,   # number of test samples to evaluate
    seed               = 3407,
    save_images        = True,
    uconn_print_option = "preprint",

    # attack parameters
    attack_method      = "diff_pgd",
    eps                = 8,           # pixel units /255
    attack_iter        = 10,
    alpha              = 1,           # PGD step size, pixel units /255

    # Diff-PGD surrogate parameters
    t                  = 20,         # SDEdit forward-diffusion timestep (0–999)
    # respace must be "" (full 1000-step model) so that q_sample and ddim_sample
    # can index timesteps up to t without going out of bounds.
    # "ddim50" would create a 50-entry schedule array, making t=150 invalid.
    # Trade-off: larger t = stronger attack but t+1 denoising steps per PGD iter.
    respace            = "",
    dm_ckpt_path       = "D:/MLSEC-Final/Diff-PGD/ckpt/256x256_diffusion_uncond.pt",
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Attack : {config['attack_method']}  eps={config['eps']}/255")
print(f"LoRA   : {config['lora_input_dir']}")


Device : cuda
Attack : diff_pgd  eps=8/255
LoRA   : logs/OSCP/unet_lora


## Cell 3 — Load Classifier

In [3]:
classifier = get_archs(config["classifier"], "uconn")
classifier.eval().to(device)

# Sanity check: classifier should produce [1, num_classes] logits
with torch.no_grad():
    dummy = torch.rand(1, 3, 224, 224, device=device)
    logits = classifier(dummy)
print(f"Classifier output shape: {logits.shape}  (expect [1, num_classes])")

Classifier output shape: torch.Size([1, 2])  (expect [1, num_classes])


## Cell 4 — Load SD 1.5 + ControlNet + LCM Scheduler + OSCP LoRA

Pipeline: `x` (image) + Canny edge map (`control_image`) -> `StableDiffusionControlNetImg2ImgPipeline` -> purified `x`.

LoRA loading follows `test.py`: strip the `"base_model.model."` prefix from PEFT keys before passing to `pipe.load_lora_weights()`.

In [4]:
# 1. ControlNet (canny edge conditioning)
controlnet = ControlNetModel.from_pretrained(
    config["controlnet_model"], torch_dtype=torch.float16
)

# 2. SD 1.5 img2img pipeline with ControlNet
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    config["pretrained_model"],
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    variant="fp16",
).to(device, dtype=torch.float32)

# 3. Swap scheduler to LCM (fast 1–5 step sampling)
if config["scheduler_type"] == "LCM":
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
else:
    raise ValueError(f"Unsupported scheduler_type: {config['scheduler_type']}")

# 4. Load OSCP-trained LoRA weights
# pipe.lora_state_dict() returns (state_dict, network_alphas); we need the first element.
# PEFT wraps keys with "base_model.model." which must be stripped for direct load.
lora_state_dict, _ = pipe.lora_state_dict(config["lora_input_dir"])
unwrapped_lora = {
    key.replace("base_model.model.", ""): weight.to(pipe.dtype)
    for key, weight in lora_state_dict.items()
}
pipe.load_lora_weights(unwrapped_lora)

print("Pipeline ready.")
print(f"  Scheduler : {pipe.scheduler.__class__.__name__}")
print(f"  LoRA keys : {len(unwrapped_lora)}")

d:\MLSEC-Final\CSC592-MLSEC-InstantPure\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 15134.37it/s]
CLIPTextModel LOAD REPORT from: C:\Users\epuls\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 35.74it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_control

Pipeline ready.
  Scheduler : LCMScheduler
  LoRA keys : 556


## Cell 4b — Load Guided Diffusion Model (Diff-PGD Surrogate)

The 256×256 unconditional guided diffusion model is used **only** during adversarial
example generation (SDEdit forward noise + DDIM reverse). It is not used in purification.


In [5]:
# Load the 256x256 unconditional guided diffusion model for the Diff-PGD surrogate.
# This model is used inside generate_x_adv_diff_pgd() only.
dm_model, diffusion = get_imagenet_dm_conf(
    device=device,
    respace=config["respace"],
    model_path=config["dm_ckpt_path"],
)
dm_model.eval()
print("Guided diffusion model loaded.")
print(f"  Checkpoint : {config['dm_ckpt_path']}")
print(f"  Respace    : {config['respace']}")


 Create DM         ---------------

 Load DM Ckpt      ---------------

Guided diffusion model loaded.
  Checkpoint : D:/MLSEC-Final/Diff-PGD/ckpt/256x256_diffusion_uncond.pt
  Respace    : 


## Cell 4c — DenoisedClassifier (SDEdit Wrapper)

`DenoisedClassifier.sdedit(x)` implements the Diff-PGD surrogate:
1. Assume `x` is in **[0, 1]**.
2. Map to `[-1, 1]` and apply `q_sample` to add `t` steps of noise.
3. Run DDIM reverse from timestep `t` back to 0.
4. Return denoised image in **[0, 1]**.

The guided diffusion model expects **256×256** input, so the image is resized
before `q_sample` and back to the original resolution after denoising.


In [ ]:
class DenoisedClassifier(torch.nn.Module):
    # Wraps the 256x256 guided diffusion model for use as a Diff-PGD surrogate.
    # sdedit(x) applies t steps of forward noise then DDIM reverse,
    # returning a denoised image in [0, 1].
    def __init__(self, classifier, diffusion, dm_model, t, device):
        super().__init__()
        self.classifier = classifier
        self.diffusion  = diffusion
        self.dm_model   = dm_model
        self.t          = t
        self.device     = device

    def sdedit(self, x, to_01=True):
        # SDEdit forward+reverse on x (expected in [0, 1]).
        # Wrapped in no_grad: the caller detaches the output before computing (to reduce memory footprint). 
        # Only the final classifier gradients are needed for Diff-PGD, not the
        # classifier gradients, so no graph through the diffusion steps is needed.
        # Without this, PyTorch retains activations for all t+1 denoising steps.
        with torch.no_grad():
            original_size = x.shape[-1]

            # guided diffusion model expects 256x256
            if original_size != 256:
                x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)

            # map [0, 1] -> [-1, 1] for the diffusion model
            x_scaled = x * 2 - 1

            t_tensor = torch.full((x_scaled.shape[0],), self.t, dtype=torch.long, device=x_scaled.device)
            x_t = self.diffusion.q_sample(x_scaled, t_tensor)

            sample = x_t
            indices = list(range(self.t + 1))[::-1]

            for i in indices:
                t_i = torch.full((x_scaled.shape[0],), i, dtype=torch.long, device=x_scaled.device)
                out = self.diffusion.ddim_sample(self.dm_model, sample, t_i)
                sample = out["sample"]

            # map [-1, 1] -> [0, 1]
            if to_01:
                sample = (sample + 1) / 2

            # resize back to original resolution
            if original_size != 256:
                sample = F.interpolate(
                    sample, size=(original_size, original_size), mode="bilinear", align_corners=False
                )

        return sample


print("DenoisedClassifier defined.")


DenoisedClassifier defined.


## Cell 5 — Helper Functions

In [7]:
def seed_everything(seed: int = 3407):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Purify image(s) via ControlNet + LCM-LoRA img2img diffusion.
def lcm_lora_denoise(x, pipe, device, config):
    batch = x.shape[0]
    prompt = [""] * batch
    original_size = x.shape[-1]  # assume square input

    # upsample to 512 if needed (pipeline expects 512x512)
    if original_size != 512:
        x = F.interpolate(x, size=(512, 512), mode="bilinear", align_corners=False)

    # build Canny edge map for ControlNet conditioning
    pil_x = TF.to_pil_image(x.squeeze(0))
    image_np = np.array(pil_x)
    canny = cv2.Canny(image_np, 100, 200)          # single-channel edge map
    canny_rgb = np.stack([canny, canny, canny], axis=2)  # replicate to 3ch
    control_image = Image.fromarray(canny_rgb)
    control_image = control_image.resize((512, 512), resample=Image.NEAREST)

    x = torch.clamp(x, 0.0, 1.0)
    generator = torch.Generator(device="cpu").manual_seed(config["seed"])

    pipe_out = pipe(
        prompt=prompt,
        image=x,
        control_image=control_image,
        num_inference_steps=config["num_inference_step"],
        guidance_scale=config["guidance_scale"],
        strength=config["strength"],
        controlnet_conditioning_scale=config["control_scale"],
        generator=generator,
        output_type="pt",
        return_dict=False,
    )

    # downsample output back to original resolution
    out_image = F.interpolate(
        pipe_out[0], size=(original_size, original_size), mode="bilinear", align_corners=False
    )
    return out_image, control_image.resize((original_size, original_size))


def generate_x_adv_diff_pgd(x, y, diffusion, dm_model, classifier, config, device):
    # Diff-PGD: attack through the guided-diffusion SDEdit surrogate.
    #
    # At each PGD step the perturbed image is first denoised via SDEdit
    # (add t-step noise, then DDIM-reverse), and the gradient is taken
    # through the classifier on that denoised image. This produces
    # adversarial examples designed to survive diffusion purification.
    # Reference: Xin et al. "Diffusion-Based Adversarial Sample Generation" (Diff-PGD).
    pgd_conf = gen_pgd_confs(
        eps=config["eps"],
        alpha=config["alpha"],
        iter=config["attack_iter"],
        input_range=(0, 1),
    )

    net = DenoisedClassifier(classifier, diffusion, dm_model, config["t"], device)

    eps   = pgd_conf["eps"]
    alpha = pgd_conf["alpha"]
    iters = pgd_conf["iter"]

    loss_fn = torch.nn.CrossEntropyLoss(reduction="sum")
    delta = torch.zeros_like(x)

    for _ in range(iters):
        # SDEdit on perturbed image (detach so gradient flows only through x_diff)
        x_diff = net.sdedit(x + delta).detach()
        x_diff.requires_grad_(True)

        with torch.enable_grad():
            loss = loss_fn(classifier(x_diff), y)
            loss.backward()
            grad_sign = x_diff.grad.data.sign()

        delta = delta + grad_sign * alpha
        delta = torch.clamp(delta, -eps, eps)

    x_adv = torch.clamp(x + delta, 0.0, 1.0)
    return x_adv.detach()


print("Helper functions defined.")


Helper functions defined.


## Cell 6 — Test Dataset + DataLoader

In [8]:
test_dataset = get_dataset("uconn", split="test", adv=False, uconn_print_option=config["uconn_print_option"])

test_loader = DataLoader(
    test_dataset,
    batch_size=1,          # test.py always uses batch_size=1 for evaluation
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

print(f"Test dataset size : {len(test_dataset)}")
print(f"Validating on     : {min(len(test_dataset), config['num_validation_set'])} samples")

Test dataset size : 1000
Validating on     : 100 samples


## Cell 7 — Evaluation Loop

| Metric | Description |
|--------|-------------|
| `classifier_accuracy` | Accuracy of raw classifier on **clean** images (no purification) |
| `attack_fail_rate` | Fraction of adversarial samples the raw classifier **still classifies correctly** (attack failure) |
| `clean_accuracy` | Accuracy after purifying **clean** images (measures purification distortion) |
| `robust_accuracy` | Accuracy after purifying **adversarial** images (the main OSCP robustness metric) |

In [9]:
# output directory
lora_tag = os.path.basename(os.path.normpath(config["lora_input_dir"]))
denoise_tag = (
    f"num_inference_step_{config['num_inference_step']}"
    f"_strength_{int(config['strength'] * 1000)}"
    f"_guidance_scale_{config['guidance_scale']}"
    f"_{config['num_validation_set']}"
    f"_control_scale_{config['control_scale']}"
)
save_path = os.path.join(
    config["output_dir"],
    config["attack_method"],
    config["classifier"],
    config["scheduler_type"],
    lora_tag,
    denoise_tag,
)

if config["save_images"]:
    for sub in ("clean_image", "adversarial_image", "robust_image", "visualization", "canny"):
        os.makedirs(os.path.join(save_path, sub), exist_ok=True)

print(f"Results will be saved to:\n  {save_path}")

# counters for metrics
classifier_accuracy = 0
attack_fail_rate    = 0
clean_accuracy      = 0
robust_accuracy     = 0
processed           = 0

seed_everything(config["seed"])

# loop
pbar = tqdm(test_loader, desc="Evaluating", total=config["num_validation_set"])

for i, (x, y) in enumerate(pbar, start=1):
    if i > config["num_validation_set"]:
        break

    x, y = x.to(device), y.to(device)

    # 1. Raw classifier accuracy on clean image
    with torch.no_grad():
        classifier_accuracy += (y == classifier(x).argmax(1)).sum().item()


    # 2. Generate adversarial example via Diff-PGD.
    # Offload the SD pipe to CPU first to free VRAM for the guided diffusion
    # surrogate + activations from t+1 denoising steps per PGD iteration.
    pipe.to("cpu")
    torch.cuda.empty_cache()
    x_adv = generate_x_adv_diff_pgd(x, y, diffusion, dm_model, classifier, config, device)
    # Restore pipe to GPU for purification
    pipe.to(device)
    torch.cuda.empty_cache()

    # 3. Attack fail rate (raw classifier on adversarial image)
    with torch.no_grad():
        attack_fail_rate += (y == classifier(x_adv).argmax(1)).sum().item()

    # 4. Purify clean image -> clean_accuracy
    with torch.no_grad():
        denoised_clean_x, natural_canny = lcm_lora_denoise(x, pipe, device, config)

    # 5. Purify adversarial image -> robust_accuracy (main OSCP metric)
    with torch.no_grad():
        robust_x, adv_canny = lcm_lora_denoise(x_adv, pipe, device, config)

    with torch.no_grad():
        clean_accuracy  += (y == classifier(denoised_clean_x.to(torch.float32)).argmax(1)).sum().item()
        robust_accuracy += (y == classifier(robust_x.to(torch.float32)).argmax(1)).sum().item()

    # save images
    if config["save_images"]:
        si(x,       save_path + f"/clean_image/{i}.png")
        si(x_adv,   save_path + f"/adversarial_image/{i}.png")
        si(robust_x, save_path + f"/robust_image/{i}.png")
        natural_canny.save(save_path + f"/canny/{i}_natural.png")
        adv_canny.save(save_path + f"/canny/{i}_adv.png")

    processed += 1
    pbar.set_postfix(
        cls_acc=f"{classifier_accuracy/processed:.3f}",
        robust=f"{robust_accuracy/processed:.3f}",
    )

print(f"\nDone. Processed {processed} samples.")

Results will be saved to:
  output_final/diff_pgd\resnet50\LCM\unet_lora\num_inference_step_5_strength_400_guidance_scale_1.0_100_control_scale_0.8


Evaluating: 100%|██████████| 100/100 [56:58<00:00, 34.19s/it, cls_acc=1.000, robust=0.990]


Done. Processed 100 samples.


## Cell 8 — Print Metrics

In [10]:
stat = {
    "classifier_accuracy" : classifier_accuracy / processed,
    "attack_fail_rate"    : attack_fail_rate    / processed,
    "clean_accuracy"      : clean_accuracy      / processed,
    "robust_accuracy"     : robust_accuracy     / processed,
}

print("\n" + "=" * 45)
print(f"  Samples evaluated : {processed}")
print(f"  Attack            : {config['attack_method']}  eps={config['eps']}/255")
print(f"  Strength          : {config['strength']}   Steps: {config['num_inference_step']}")
print("=" * 45)
for k, v in stat.items():
    print(f"  {k:<26}: {v:.4f}  ({v*100:.1f}%)")
print("=" * 45)


  Samples evaluated : 100
  Attack            : diff_pgd  eps=8/255
  Strength          : 0.4   Steps: 5
  classifier_accuracy       : 1.0000  (100.0%)
  attack_fail_rate          : 0.0000  (0.0%)
  clean_accuracy            : 0.9900  (99.0%)
  robust_accuracy           : 0.9900  (99.0%)


## Cell 9 — Save CSV

In [11]:
os.makedirs(save_path, exist_ok=True)
csv_path = os.path.join(save_path, "stat.csv")

df = pd.DataFrame(stat, index=[0])
df.to_csv(csv_path, index=False)

print(f"Results saved at: {csv_path}")
print(df.to_string(index=False))

Results saved at: output_final/diff_pgd\resnet50\LCM\unet_lora\num_inference_step_5_strength_400_guidance_scale_1.0_100_control_scale_0.8\stat.csv
 classifier_accuracy  attack_fail_rate  clean_accuracy  robust_accuracy
                 1.0               0.0            0.99             0.99
